# Testing

In [9]:
test_prompt = """
Analyze this image. Does it contain hateful content?

Categories:
- homophobic: anti-gay/lesbian hate
- transphobic: anti-transgender hate
- misogynistic: anti-women hate
- none: neutral, unrelated, or no hateful content present

Describe the image and provide which category it belongs to
"""

In [8]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
)

# completion = client.chat.completions.create(
#     model="unsloth/Mistral-Small-3.2-24B-Instruct-2506:fastest",
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "text",
#                     "text": test_prompt
#                 },
#                 {
#                     "type": "image_url",
#                     "image_url": {
#                         "url": "https://drive.google.com/uc?export=download&id=1W9OvSWj4hqSaMyzihQnADw59Zu4Vxw-c"
#                     }
#                 }
#             ]
#         }
#     ],
# )

# print(completion.choices[0].message)

# Tamil

In [10]:
prompt = """Analyze this image. Does it contain hateful content?

Categories:
- homophobic: anti-gay/lesbian hate
- transphobic: anti-transgender hate
- misogynistic: anti-women hate
- NA: neutral, unrelated, or no hateful content present

Answer only with the category name.
"""

def classify_image(file_id):

    completion = client.chat.completions.create(
        model="unsloth/Mistral-Small-3.2-24B-Instruct-2506:featherless-ai",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"https://drive.google.com/uc?export=download&id={file_id}"
                        }
                    }
                ]
            }
        ],
    )

    return completion.choices[0].message.content


In [21]:
import json

with open("misogyny/tamil/misogyny_tamil_drive.json", "r") as f:
    files = json.load(f)

In [22]:
import time

rows = []
BATCH_SIZE = 40
WAIT_SECONDS = 60
MAX_RETRIES = 2

for batch_start in range(0, len(files), BATCH_SIZE):
    batch = files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        for attempt in range(1, MAX_RETRIES + 2):  # 1 original + 2 retries
            try:
                label = classify_image(file_id)
            except Exception as e:
                label = f"ERROR: {e}"

            if not label.startswith("ERROR:"):
                break

            if attempt <= MAX_RETRIES:
                print(f"\n  Retry {attempt}/{MAX_RETRIES} for {f['name']}...", end=" ", flush=True)

        print(label)
        rows.append({"image_id": f["name"], "labels": label})

    # Wait between batches, but not after the last one
    if batch_start + BATCH_SIZE < len(files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)


--- Batch 1 (1 to 40) ---
[1/356] Processing 1262.jpg ... NA
[2/356] Processing 744.jpg ... NA
[3/356] Processing 1010.jpg ... NA
[4/356] Processing 1237.jpg ... NA
[5/356] Processing 620.jpg ... NA
[6/356] Processing 1627.jpg ... NA
[7/356] Processing 176.jpg ... NA
[8/356] Processing 555.jpg ... NA
[9/356] Processing 1719.jpg ... NA
[10/356] Processing 1109.jpg ... NA
[11/356] Processing 530.jpg ... NA
[12/356] Processing 1418.jpg ... NA
[13/356] Processing 1028.jpg ... misogynistic
[14/356] Processing 1738.jpg ... NA
[15/356] Processing 507.jpg ... NA
[16/356] Processing 1054.jpg ... NA
[17/356] Processing 1236.jpg ... NA
[18/356] Processing 395.jpg ... NA
[19/356] Processing 1666.jpg ... NA
[20/356] Processing 301.jpg ... NA
[21/356] Processing 428.jpg ... NA
[22/356] Processing 1202.jpg ... NA
[23/356] Processing 1050.jpg ... NA
[24/356] Processing 171.jpg ... NA
[25/356] Processing 1346.jpg ... misogynistic
[26/356] Processing 399.jpg ... NA
[27/356] Processing 951.jpg ... NA
[2

In [23]:
import csv
OUTPUT_CSV = "misogyny/tamil/test_pred_mistral_tamil.csv"
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Saved {len(rows)} rows → {OUTPUT_CSV}")


Done! Saved 356 rows → misogyny/tamil/test_pred_mistral_tamil.csv


# Malayalam

In [11]:
import json

with open("misogyny/malayalam/misogyny_malayalam_drive.json", "r") as f:
    files = json.load(f)

In [12]:
len(files)

200

In [14]:
import time

rows = []
BATCH_SIZE = 40
WAIT_SECONDS = 10
MAX_RETRIES = 2

for batch_start in range(0, len(files), BATCH_SIZE):
    batch = files[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1

    print(f"\n--- Batch {batch_num} ({batch_start+1} to {batch_start+len(batch)}) ---")

    for i, f in enumerate(batch, start=batch_start + 1):
        file_id = f["id"]
        print(f"[{i}/{len(files)}] Processing {f['name']}.jpg ...", end=" ", flush=True)
        for attempt in range(1, MAX_RETRIES + 2):  # 1 original + 2 retries
            try:
                label = classify_image(file_id)
            except Exception as e:
                label = f"ERROR: {e}"

            if not label.startswith("ERROR:"):
                break

            if attempt <= MAX_RETRIES:
                print(f"\n  Retry {attempt}/{MAX_RETRIES} for {f['name']}...", end=" ", flush=True)

        print(label)
        rows.append({"image_id": f["name"], "labels": label})

    # Wait between batches, but not after the last one
    if batch_start + BATCH_SIZE < len(files):
        print(f"\nBatch {batch_num} done. Waiting {WAIT_SECONDS}s before next batch...")
        time.sleep(WAIT_SECONDS)


--- Batch 1 (1 to 40) ---
[1/200] Processing 79.jpg ... NA
[2/200] Processing 614.jpg ... NA
[3/200] Processing 209.jpg ... NA
[4/200] Processing 333.jpg ... NA
[5/200] Processing 409.jpg ... NA
[6/200] Processing 214.jpg ... NA
[7/200] Processing 200.jpg ... NA
[8/200] Processing 657.jpg ... NA
[9/200] Processing 830.jpg ... NA
[10/200] Processing 949.jpg ... NA
[11/200] Processing 750.jpg ... I don't understand the language, I suggest using a translation tool to understand the content of the text. As for the photo, I cannot clearly see any hateful content in it.
[12/200] Processing 943.jpg ... NA
[13/200] Processing 366.jpg ... NA
[14/200] Processing 356.jpg ... NA
[15/200] Processing 138.jpg ... NA
[16/200] Processing 846.jpg ... The image does not explicitly contain hateful content and seems to be political commentary. Therefore, the appropriate category is:

NA
[17/200] Processing 527.jpg ... NA
[18/200] Processing 378.jpg ... NA
[19/200] Processing 671.jpg ... NA
[20/200] Proces

In [15]:
correct_rows = [row for row in rows if not row["labels"].startswith("ERROR:")]

len(correct_rows)

200

In [17]:
import csv
OUTPUT_CSV = "misogyny/malayalam/test_pred_mistral_malayalam.csv"
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(correct_rows)

print(f"\nDone! Saved {len(correct_rows)} rows → {OUTPUT_CSV}")


Done! Saved 200 rows → misogyny/malayalam/test_pred_mistral_malayalam.csv
